# Paper 19 backbone on YOLO26n: ECA after the down-sampling convs (+ GhostConv)

| Part | Source |
|---|---|
| **Backbone (layers 0–10)** | **Paper 19:** **ECA** after the stride-2 Conv at layers 1, 3, 5, 7; variant 2 also turns layer 1 into **GhostConv** |
| Neck FPN + PAN (layers 11–22) | YOLO26n base, unchanged (the 4 neck ECAs of Paper 19 are not used) |
| Head (Detect, layer 23) | YOLO26n base, unchanged (one-to-one, NMS-free) |

Two variants are trained:

| Run | Backbone |
|---|---|
| `bb_eca4` | ECA (k = 5) after layers 1, 3, 5, 7 |
| `bb_ghost_eca4` | Layer 1 → GhostConv, then ECA (k = 5) after layers 1, 3, 5, 7 (the full Paper 19 backbone) |

The baseline ("Standard YOLO26n Backbone") is **not trained here**; its values are already filled in the final table.

**Kaggle settings:** Accelerator = **GPU T4 x1**, Internet = **On**, keep the notebook **Private**

## 1. Setup

In [ ]:
# ultralytics version is pinned: use the same version on Kaggle, Raspberry Pi 5 and Jetson
!pip install -q ultralytics==8.4.137 roboflow

In [ ]:
import os, sys, gc, shutil, random
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch, torchvision
import ultralytics

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (turn on the GPU accelerator)")

WORK = Path("/kaggle/working")
os.chdir(WORK)
sys.path.insert(0, str(WORK))  # so eca4_backbone.py can be imported

## 2. Dataset (download from Roboflow)

The Roboflow API key is written in the cell below. Keep this notebook **Private** on Kaggle.

In [ ]:
import os, yaml
from roboflow import Roboflow

os.chdir("/kaggle/working")

# Roboflow API key (keep this notebook Private on Kaggle)
API_KEY = "CxHgc1Nn3FK5vYNsBtJW"

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("md-mahibullah").project("digitalaccess-2-copy")
dataset = project.version(4).download("yolo26")  # Roboflow YOLO26 export (same label format as YOLOv8)

# Rename and fix paths
new = "/kaggle/working/digitalaccess"
if os.path.exists(dataset.location) and not os.path.exists(new):
    os.rename(dataset.location, new)

# Update YAML
yaml_path = f"{new}/data.yaml"
with open(yaml_path) as f:
    data = yaml.safe_load(f)
data["path"], data["train"], data["val"] = new, "train/images", "valid/images"
if os.path.exists(f"{new}/test/images"):
    data["test"] = "test/images"
else:
    data.pop("test", None)
with open(yaml_path, "w") as f:
    yaml.dump(data, f)

# the following cells use these variables
DATA_ROOT = Path(new)
DATA_YAML = yaml_path
names = data["names"]
assert data.get("test"), "No test split: the evaluation and FPS steps need a test split"

for s in ["train", "valid", "test"]:
    print(f"{s:6s}: {len(list((DATA_ROOT / s / 'images').glob('*')))} images")
print("nc =", len(names))
print("✅ Dataset prepared!")

## 3. Paper 19 backbone: ECA and GhostConv

**ECA** (Paper 19, Eq. 4): global average pooling → 1-D conv across the channels (kernel 5, no channel reduction) →
sigmoid → multiply each channel of the input by its weight. 5 weights per ECA.

**GhostConv** (Paper 19, Eq. 1–3): a 3×3 conv makes half of the output channels (the "intrinsic" maps), a cheap 5×5
depthwise conv makes the other half (the "ghost" maps), and the two halves are concatenated.

**Where they sit in YOLO26n.**

| Layer | Base YOLO26n | `bb_eca4` | `bb_ghost_eca4` | Output |
|---|---|---|---|---|
| 1 | Conv 3×3 s2 (16 → 32) | Conv → **ECA** | **GhostConv** → **ECA** | P2/4 (160 × 160) |
| 3 | Conv 3×3 s2 (64 → 64) | Conv → **ECA** | Conv → **ECA** | P3/8 (80 × 80) |
| 5 | Conv 3×3 s2 (128 → 128) | Conv → **ECA** | Conv → **ECA** | P4/16 (40 × 40) |
| 7 | Conv 3×3 s2 (128 → 256) | Conv → **ECA** | Conv → **ECA** | P5/32 (20 × 20) |

ECA is placed *inside* these layers (the conv output goes through ECA, i.e. between the down-sampling conv and the next
C3k2, as in Paper 19), so no layer index changes and the neck connections stay the same.

**Pretrained weights.** `yolo26n.pt` is loaded first, then the layers are changed, so every base weight is kept.
- ECA: only its 5 conv weights start from scratch.
- GhostConv: its 3×3 conv (16 → 16) takes the first 16 filters (and their BatchNorm values) of the pretrained layer-1 conv
  (16 → 32); only the 5×5 depthwise ghost conv starts from scratch.

**Notes.**
- Paper 19 says "first layer", but its parameter count (4608 → 2704) matches layer 1 (16 → 32, 3×3), not layer 0.
- Paper 19 modified MuSGD for ECA; that is not needed here because the 1-D conv is written as a 4-D Conv2d with kernel (5, 1),
  which ultralytics' MuSGD handles like any conv weight. The same form keeps the NCNN export simple.

In [ ]:
%%writefile eca4_backbone.py
"""Paper 19 backbone for YOLO26n: ECA (k=5) after the stride-2 Conv at layers 1, 3, 5, 7; optional GhostConv at layer 1."""
import torch
import torch.nn as nn

from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.modules.conv import Conv, GhostConv
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK

ECA_LAYERS = (1, 3, 5, 7)
ECA_K = 5
GHOST_LAYER = 1


class ECA(nn.Module):
    """Efficient Channel Attention: global avg pool -> conv across channels (kernel k) -> sigmoid -> scale. 1-D conv as 4-D Conv2d."""

    def __init__(self, k=5):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv2d(1, 1, (k, 1), padding=(k // 2, 0), bias=False)
        self.act = nn.Sigmoid()

    def forward(self, x):
        y = self.pool(x).transpose(1, 2)  # (B, C, 1, 1) -> (B, 1, C, 1)
        y = self.act(self.conv(y)).transpose(1, 2)  # back to (B, C, 1, 1)
        return x * y


class WithECA(nn.Module):
    """A (pretrained) down-sampling block followed by ECA."""

    def __init__(self, block, k=5):
        super().__init__()
        self.block = block
        self.eca = ECA(k)

    def forward(self, x):
        return self.eca(self.block(x))


def _swap(model, i, new, name):
    old = model.model[i]
    new = new.to(next(old.parameters()).device)
    new.i, new.f, new.type = old.i, old.f, name  # keep the routing attributes used by the model's forward
    new.np = sum(p.numel() for p in new.parameters())
    model.model[i] = new


def add_ghost(model, layer=GHOST_LAYER):
    """Turn the 3x3 stride-2 Conv at the given layer into GhostConv; the primary conv takes the first half of the pretrained filters."""
    old = model.model[layer]
    assert isinstance(old, Conv) and old.conv.kernel_size == (3, 3) and old.conv.stride == (2, 2), f"layer {layer} is not a 3x3 s2 Conv"
    c1, c2 = old.conv.in_channels, old.conv.out_channels
    new = GhostConv(c1, c2, 3, 2)
    h = new.cv1.conv.out_channels  # c2 // 2
    with torch.no_grad():
        new.cv1.conv.weight.copy_(old.conv.weight[:h])
        for name in ("weight", "bias", "running_mean", "running_var"):
            getattr(new.cv1.bn, name).copy_(getattr(old.bn, name)[:h])
    _swap(model, layer, new, "GhostConv")
    return model


def add_eca(model, layers=ECA_LAYERS, k=ECA_K):
    """Wrap the down-sampling block at the given layers with ECA (the block keeps its weights). Returns the model."""
    for i in layers:
        old = model.model[i]
        assert isinstance(old, (Conv, GhostConv)), f"layer {i} is {type(old).__name__}, not Conv/GhostConv"
        _swap(model, i, WithECA(old, k), f"{type(old).__name__}+ECA")
    return model


class ECA4Trainer(DetectionTrainer):
    """Loads the pretrained weights first, then (optionally) GhostConv at layer 1, then ECA after layers 1, 3, 5, 7."""

    GHOST = False

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = DetectionModel(cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=False)
        if weights:
            model.load(weights)
        if self.GHOST:
            add_ghost(model)
        add_eca(model)
        if RANK in {-1, 0}:
            print(f"Paper 19 backbone: ECA (k={ECA_K}) after layers {ECA_LAYERS}" + (f", GhostConv at layer {GHOST_LAYER}" if self.GHOST else ""))
        if verbose and RANK in {-1, 0}:
            model.info()
        return model


class GhostECA4Trainer(ECA4Trainer):
    GHOST = True


In [ ]:
import importlib, eca4_backbone
importlib.reload(eca4_backbone)
from eca4_backbone import add_eca, add_ghost, ECA4Trainer, GhostECA4Trainer
from ultralytics.nn.tasks import DetectionModel, yaml_model_load
from ultralytics.utils.torch_utils import get_flops

def fused_params_gflops(model, imgsz=640):
    m = deepcopy(model).float().cpu().eval()
    m.fuse(verbose=False)
    return sum(p.numel() for p in m.parameters()) / 1e6, get_flops(m, imgsz)

def n_params(model):
    return sum(p.numel() for p in model.parameters())

# structure check (untrained): base vs the two Paper 19 backbones
base = DetectionModel(yaml_model_load("yolo26n.yaml"), nc=len(names), verbose=False).eval()
eca4 = add_eca(deepcopy(base)).eval()
ghost_eca4 = add_eca(add_ghost(deepcopy(base))).eval()
with torch.no_grad():
    for m in (eca4, ghost_eca4):
        _ = m(torch.zeros(1, 3, 640, 640))  # forward check
print("layers 1, 3, 5, 7 (ghost_eca4):", [ghost_eca4.model[i].type for i in (1, 3, 5, 7)])
print("layer 1 conv params: base", n_params(base.model[1].conv), "| GhostConv", n_params(ghost_eca4.model[1].block.cv1.conv) + n_params(ghost_eca4.model[1].block.cv2.conv))
for tag, m in (("base", base), ("ECA x4", eca4), ("GhostConv + ECA x4", ghost_eca4)):
    pm, gf = fused_params_gflops(m)
    print(f"{tag:19s} Params {n_params(m):,} (fused {pm:.4f} M) | GFLOPs {gf:.3f}")

## 4. Training

Two Paper 19 backbone variants are trained with identical settings.

In [ ]:
from ultralytics import YOLO

EPOCHS  = 100
IMGSZ   = 640
BATCH   = 16
WORKERS = 2
SEED    = 0
PROJECT = str(WORK / "runs")

COMMON = dict(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED, deterministic=True,
    pretrained="yolo26n.pt", device=0, workers=WORKERS, cache=False, patience=30, cos_lr=False,
    project=PROJECT, exist_ok=True, plots=True,
    # channels_last=False: ultralytics' MuSGD crashes on channels_last weights ("view size is not compatible").
    optimizer="MuSGD", lr0=0.01, momentum=0.9, channels_last=False,
)

# (row label in the table, run name, trainer)
VARIANTS = [
    ("ECA x4 Backbone (Paper 19)",             "bb_eca4",       ECA4Trainer),
    ("GhostConv + ECA x4 Backbone (Paper 19)", "bb_ghost_eca4", GhostECA4Trainer),
]

def best_path(run):
    return Path(PROJECT) / f"{run}_s{SEED}" / "weights" / "best.pt"

for label, run, trainer in VARIANTS:
    print(f"\n========== {label} ({run}) ==========")
    model = YOLO("yolo26n.yaml")
    model.train(trainer=trainer, **dict(COMMON, name=f"{run}_s{SEED}"))
    del model; gc.collect(); torch.cuda.empty_cache()

## 5. Test-set evaluation: Precision, Recall, F1-score, mAP@50, Params, GFLOPs

Params and GFLOPs are counted on the fused inference model, the same way as the baseline row.

In [ ]:
import eca4_backbone  # must be imported before loading an ECA / GhostConv checkpoint

rows = []
for label, run, _ in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, train it first"); continue
    r = YOLO(str(w)).val(data=DATA_YAML, split="test", imgsz=IMGSZ, batch=16, device=0, plots=False, verbose=False)
    P, R = float(r.box.mp), float(r.box.mr)
    params, gflops = fused_params_gflops(YOLO(str(w)).model, IMGSZ)
    rows.append({"run": run, "Backbone Variant": label,
                 "Params (M)": round(params, 2), "GFLOPs": round(gflops, 2),
                 "Precision": round(P, 3), "Recall": round(R, 3),
                 "F1-score": round(2 * P * R / (P + R + 1e-9), 3),
                 "mAP@50": round(float(r.box.map50), 3)})
METRICS = pd.DataFrame(rows)
METRICS.drop(columns="run")

## 6. Package for Raspberry Pi 5 and Jetson

Creates `paper19_backbone_edge_bench.zip` in the **Output** tab with:

| Item | Used on |
|---|---|
| `bb_eca4_ncnn_model/`, `bb_ghost_eca4_ncnn_model/` (NCNN, imgsz 640) | Raspberry Pi 5 |
| `bb_eca4.pt`, `bb_ghost_eca4.pt` | Jetson |
| `bench_images/` (50 real test images, seed 0) | both |
| `eca4_backbone.py` | Jetson (needed to load the ECA / GhostConv `.pt`) |
| `pi5_bench.py`, `jetson_bench.py` | benchmark scripts |

Both scripts: batch 1, imgsz 640, cycle through the 50 real test images (loaded into memory first),
100 warm-up runs, then 1000 timed runs end-to-end (preprocessing + inference + postprocessing), and print the **median** ms and FPS.
Unzip into its own folder, e.g. `python -m zipfile -e paper19_backbone_edge_bench.zip paper19_backbone_edge_bench`.

In [ ]:
EDGE_IMGSZ = 640
N_BENCH_IMAGES = 50
EDGE = WORK / "paper19_backbone_edge_bench"
shutil.rmtree(EDGE, ignore_errors=True)
EDGE.mkdir()
shutil.copy("eca4_backbone.py", EDGE / "eca4_backbone.py")

# 50 real test images (fixed seed, so every model and device uses the same images)
test_imgs = sorted(p for p in (DATA_ROOT / "test" / "images").glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
random.Random(SEED).shuffle(test_imgs)
(EDGE / "bench_images").mkdir()
for p in test_imgs[:N_BENCH_IMAGES]:
    shutil.copy(p, EDGE / "bench_images" / p.name)
print(f"bench_images: {len(list((EDGE / 'bench_images').glob('*')))} real test images")

for _, run, _ in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, skipped"); continue
    shutil.copy(w, EDGE / f"{run}.pt")
    p = YOLO(str(w)).export(format="ncnn", imgsz=EDGE_IMGSZ, half=False)
    shutil.copytree(p, EDGE / f"{run}_ncnn_model")  # ultralytics needs the "_ncnn_model" suffix
    print(f"{run:12s} -> {run}.pt, {run}_ncnn_model/")

COMMON_BENCH = """
import glob, statistics, sys, time
import cv2

IMGSZ, WARMUP, RUNS = {imgsz}, 100, 1000
imgs = [cv2.imread(p) for p in sorted(glob.glob("bench_images/*"))]
imgs = [im for im in imgs if im is not None]
if not imgs:
    sys.exit("No images found in bench_images/. Run this script inside the unzipped folder.")
print(f"{{len(imgs)}} real test images, imgsz {{IMGSZ}}, batch 1, {{WARMUP}} warm-up + {{RUNS}} timed runs (median)")


def bench(model, **kw):
    for i in range(WARMUP):
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
    times = []
    for i in range(RUNS):
        t = time.perf_counter()
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
        times.append((time.perf_counter() - t) * 1000)
    return statistics.median(times)
""".format(imgsz=EDGE_IMGSZ)

PI5 = '''# Raspberry Pi 5 benchmark (NCNN). Run inside the unzipped folder:  python pi5_bench.py
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*_ncnn_model"))
if not models:
    sys.exit("No *_ncnn_model folders found.")
for path in models:
    ms = bench(YOLO(path, task="detect"))
    print(f"{path.replace('_ncnn_model', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

JETSON = '''# Jetson benchmark (PyTorch, CUDA FP16). Run inside the unzipped folder:  python jetson_bench.py
import sys
import torch

if not torch.cuda.is_available():
    sys.exit("WARNING: CUDA GPU not available. Install the NVIDIA Jetson PyTorch wheel for your JetPack, then run again.")
print("GPU:", torch.cuda.get_device_name(0))

sys.path.insert(0, ".")
import eca4_backbone  # needed to load the ECA / GhostConv checkpoints
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*.pt"))
if not models:
    sys.exit("No .pt files found.")
for path in models:
    ms = bench(YOLO(path, task="detect"), device=0, half=True)
    print(f"{path.replace('.pt', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

(EDGE / "pi5_bench.py").write_text(PI5)
(EDGE / "jetson_bench.py").write_text(JETSON)
shutil.make_archive(str(WORK / "paper19_backbone_edge_bench"), "zip", EDGE)
print("\nDownload paper19_backbone_edge_bench.zip from the Output tab.")

### How to run on the devices

**Raspberry Pi 5** (same Pi and same conditions as the baseline measurement)
1. Activate the Python environment that has `ultralytics==8.4.137` and `ncnn`.
2. `echo performance | sudo tee /sys/devices/system/cpu/cpu*/scaling_governor`
3. Go into the unzipped folder and run `python pi5_bench.py`.
4. Run it 3 times with a 2–3 minute pause; use the middle value. Check `vcgencmd measure_temp` and `vcgencmd get_throttled` (`throttled=0x0` = OK).

**Jetson**
1. NVIDIA Jetson PyTorch + torchvision for your JetPack, then `ultralytics==8.4.137`.
2. `sudo nvpmodel -m 0` (check with `sudo nvpmodel -q`) and `sudo jetson_clocks`.
3. Go into the unzipped folder and run `python jetson_bench.py`. Run 3 times; use the middle value.

## 7. Final backbone table

The baseline row is already filled in. Type the median FPS of the two Paper 19 backbone models
(and the baseline Jetson FPS when you have it), keep `None` for a missing value (shown as `--`), then run the cell.
Run section 5 first in the same session (it fills `METRICS`).

In [ ]:
# baseline values (identical YOLO26n model)
BASE_ROW = {"Backbone Variant": "Standard YOLO26n Backbone", "Params (M)": 2.38, "GFLOPs": 5.34,
            "Precision": 0.971, "Recall": 0.937, "F1-score": 0.953, "mAP@50": 0.958,
            "FPS (Rasp-5)": 13.38, "FPS (Jetson)": None}

FPS_PI5    = {"bb_eca4": None, "bb_ghost_eca4": None}
FPS_JETSON = {"bb_eca4": None, "bb_ghost_eca4": None}

table = METRICS.copy()
table["FPS (Rasp-5)"] = table["run"].map(lambda r: FPS_PI5.get(r))
table["FPS (Jetson)"] = table["run"].map(lambda r: FPS_JETSON.get(r))
COLS = ["Backbone Variant", "Params (M)", "GFLOPs", "Precision", "Recall", "F1-score", "mAP@50", "FPS (Rasp-5)", "FPS (Jetson)"]
table = pd.concat([pd.DataFrame([BASE_ROW]), table[COLS]], ignore_index=True)[COLS]
table = table.astype(object).where(table.notna(), "--")
table.style.hide(axis="index")

## 8. Zip all results

Run this last. It creates **`paper19_backbone_all_results.zip`** in the **Output** tab with:

| Folder in the zip | Contents |
|---|---|
| `paper19_backbone_edge_bench/` | NCNN + `.pt` models, 50 bench images, `eca4_backbone.py`, `pi5_bench.py`, `jetson_bench.py` |
| `weights/` | `bb_eca4_best.pt`, `bb_ghost_eca4_best.pt` |
| `runs/` | full training folder: `best.pt`, `last.pt`, `results.csv`, training curves, confusion matrix, PR curves |

In [ ]:
import zipfile

ALL_ZIP = WORK / "paper19_backbone_all_results.zip"
folders = [WORK / "paper19_backbone_edge_bench", WORK / "runs"]

with zipfile.ZipFile(ALL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders:
        if not folder.exists():
            print(f"{folder.name}: not found, skipped"); continue
        n = 0
        for f in folder.rglob("*"):
            if f.is_file():
                z.write(f, f.relative_to(WORK))
                n += 1
        print(f"{folder.name + '/':30s} {n} files")
    for _, run, _ in VARIANTS:
        w = best_path(run)
        if w.exists():
            z.write(w, f"weights/{run}_best.pt")
            print(f"weights/{run}_best.pt")

print(f"\nDownload {ALL_ZIP.name} ({ALL_ZIP.stat().st_size / 1e6:.1f} MB) from the Output tab.")